# 04 · The predictive agent: Jev decides, TabPFN predicts, Claude talks

Everything from the previous steps, in one loop:

```
question ──▶ Jev: prediction / lookup / other?   (a decision model — probabilities, ~100 ms)
                │
   prediction ──┼─▶ Claude + TabPFN tools         (the handoff from step 03, as Python tools)
   lookup ──────┼─▶ Claude + a pandas tool
   other ───────┴─▶ Claude, no tools
```

Each model does the thing it's for:

- **Jev** (TypeSafe's decision model) makes the routing decision. It doesn't generate text — you give it the question and a typed `Choice`, and it returns a probability for each option. Optional: without a `TYPESAFE_API_KEY` the agent falls back to the Claude classifier from notebook 02.
- **TabPFN** makes the predictions, via the Prior Labs API.
- **Claude** does the analyst's job: writes the pandas, calls the tools, explains the result.

The code is in `agent/` — small enough to read in five minutes. This notebook walks through it.

In [1]:
# Run from the repo root or from notebooks/ — this makes `agent` importable either way
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "agent").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Keys: found in your environment or .env → used as-is; otherwise you're prompted (input hidden, nothing saved to disk)
from agent.keys import ensure_keys
ensure_keys("TABPFN_TOKEN", "ANTHROPIC_API_KEY", optional=("TYPESAFE_API_KEY",))
print("Jev router:", "on" if os.environ.get("TYPESAFE_API_KEY") else "off (falls back to Claude)")

TABPFN_TOKEN       found in environment
ANTHROPIC_API_KEY  found in environment
TYPESAFE_API_KEY   found in environment
Jev router: on


## 1. The router

Here's the whole Jev call. `state` is the input, `questions` is a dict of typed questions, and the answer carries a full distribution.

In [3]:
from agent.router import route, TASK_TYPES

for q in [
    "How many customers churned?",
    "Which customers are most likely to churn next month?",
    "Which customers will probably churn — I mean the ones who already did, how many?",
    "Thanks!",
]:
    r = route(q, router="jev")
    probs = "  ".join(f"{k}: {v:.0%}" for k, v in sorted(r.probabilities.items(), key=lambda kv: -kv[1]))
    print(f"[{r.router}] {r.task_type:<11} {probs:<45}  ← {q}")

[jev] lookup      lookup: 100%  prediction: 0%  other: 0%        ← How many customers churned?
[jev] prediction  prediction: 100%  lookup: 0%  other: 0%        ← Which customers are most likely to churn next month?
[jev] lookup      lookup: 79%  prediction: 21%  other: 0%        ← Which customers will probably churn — I mean the ones who already did, how many?
[jev] other       other: 100%  prediction: 0%  lookup: 0%        ← Thanks!


Notice the third one: with Jev, the ambiguous question should come back with the probability mass *split*, instead of a confident wrong label. The agent uses that — below a confidence threshold it gives Claude both tool sets and lets it ask a clarifying question rather than committing.

If you have both keys, compare the two routers on the same questions: Jev returns a distribution, Claude returns a label.

In [4]:
import time
q = "Score our customer base for churn risk."
for router in (["jev"] if os.environ.get("TYPESAFE_API_KEY") else []) + ["claude"]:
    t0 = time.perf_counter(); r = route(q, router=router); dt = time.perf_counter() - t0
    print(f"{router:<7} {r.task_type:<11} {r.probabilities}   {dt*1000:.0f} ms")

jev     prediction  {'prediction': 1.0, 'lookup': 0.0, 'other': 0.0}   278 ms
claude  prediction  {'prediction': 1.0}   3621 ms


## 2. The tools

Two tool shapes. The prediction tools wrap a `ChurnModel` that fits TabPFN on the history split **once** and caches it; the lookup tool runs a pandas snippet Claude writes. Open `agent/tools.py` to see them — the important part is what's *not* there: no model selection, no tuning, no training loop.

In [5]:
from agent.tools import ChurnModel, run_lookup

print(run_lookup("print(df.groupby('Contract')['Churn'].apply(lambda s: (s=='Yes').mean()).round(3))"))

Contract
Month-to-month    0.426
One year          0.108
Two year          0.028
Name: Churn, dtype: float64


In [6]:
churn = ChurnModel()
out = churn.score_current_customers(top_n=5)      # first call fits TabPFN (a few seconds), later calls reuse it
print(f"fit in {churn.fit_seconds:.1f}s · scored {out['n_scored']} customers in {out['seconds']}s · {out['share_above_50pct']:.0%} above 50%")
import pandas as pd; pd.DataFrame(out["top_customers"])

fit in 2.0s · scored 1409 customers in 4.6s · 20% above 50%


,row,tenure,Contract,InternetService,PaymentMethod,MonthlyCharges,p_churn
0,307,1,Month-to-month,Fiber optic,Electronic check,94.00,0.939
1,2,1,Month-to-month,Fiber optic,Electronic check,74.45,0.911
2,249,1,Month-to-month,Fiber optic,Electronic check,93.30,0.910
3,96,1,Month-to-month,Fiber optic,Electronic check,89.30,0.910
4,914,1,Month-to-month,Fiber optic,Electronic check,72.10,0.908


## 3. The agent

`PredictiveAgent.ask()` routes, picks the tool set, then runs a standard Claude tool-use loop until Claude stops calling tools. `verbose=True` prints the route and every tool call so you can see the handoff happen.

In [8]:
from agent import PredictiveAgent
agent = PredictiveAgent()          # router="auto": Jev if the key is set, else Claude
agent.churn = churn                # reuse the fit from above

turn = agent.ask("How many customers do we have, and what share of them churned?")
print(turn.answer)

[router=jev] lookup  (lookup 100%, prediction 0%, other 0%)
[tool] run_pandas({"code": "print(len(df)); print(df['Churn'].value_counts()); print((df['Churn']=='Yes').mean())"})
The history has **5,634 labeled customers**. **1,495 of them (26.5%) churned**, and the other 4,139 stayed.

This is the base churn rate. It's the benchmark for any customer or segment: a churn figure well above 26.5% means higher-than-typical risk, and a figure well below it means lower risk.


In [9]:
turn = agent.ask("Which of our current customers are most likely to churn? Give me the top 10.")
print(turn.answer)

[router=jev] prediction  (prediction 100%, lookup 0%, other 0%)
[tool] score_current_customers({"top_n": 10})
Here are the 10 current customers most likely to churn, scored by TabPFN across all 1,409 current customers:

| Row | Tenure (months) | Monthly charges | p(churn) |
|---|---|---|---|
| 307 | 1 | $94.00 | 93.9% |
| 2 | 1 | $74.45 | 91.1% |
| 249 | 1 | $93.30 | 91.0% |
| 96 | 1 | $89.30 | 91.0% |
| 914 | 1 | $72.10 | 90.8% |
| 1245 | 1 | $69.55 | 89.8% |
| 211 | 1 | $69.85 | 89.4% |
| 1086 | 1 | $90.55 | 89.3% |
| 572 | 1 | $78.95 | 88.4% |
| 1104 | 1 | $80.55 | 88.4% |

All ten share the same profile. They are in their first month, on a month-to-month contract, on fiber optic, and pay by electronic check.

**How to read it:** Each p(churn) is the model's estimated chance that the customer leaves. The historical base rate is 26.5%, so these customers are roughly 3.3 to 3.5 times as likely to churn as a typical customer. About 20% of all current customers score above 50%, so the a

In [10]:
# A single customer, pasted as a row. (This is the last row of data/telco_churn_current.csv — the real answer is Yes.)
turn = agent.ask("""Is this customer likely to churn?
Female,0,Yes,Yes,1,Yes,Yes,'Fiber optic',Yes,No,Yes,No,Yes,Yes,Month-to-month,Yes,'Electronic check',102.45,102.45""")
print(turn.answer)

[router=jev] prediction  (prediction 99%, lookup 1%, other 0%)
[tool] score_rows({"rows_csv": "Female,0,Yes,Yes,1,Yes,Yes,'Fiber optic',Yes,No,Yes,No,Yes,Yes,Month-to-month,Yes,'Electronic check',102.4)
Yes, this customer is likely to churn. TabPFN gives a churn probability of **85.3%**, so the label is "Yes" at the 0.5 cut-off.

For comparison, the base churn rate in the history is 26.5%. This customer's risk is about 3 times that.

The profile has several features that are often linked to churn. It is a 1-month tenure on a month-to-month contract, with fiber optic, electronic check payment and high monthly charges of $102.45. I'm reading those from the row you pasted. The score itself comes from TabPFN, not from these features individually.


In [ ]:
turn = agent.ask("What's the average monthly charge for customers who churned vs. those who didn't?")
print(turn.answer)

In [ ]:
# The ambiguous one — watch what the router does and whether the agent commits or asks.
turn = agent.ask("Which customers will probably churn — I mean the ones who already did, how many?")
print(turn.answer)

## 4. Your turn

Try your own questions. A few ideas:

- *"Score these three customers"* + three rows copied from `data/telco_churn_current.csv`
- *"Which contract type has the highest churn, and which current customers on it are at most risk?"* — one question, both kinds of work
- *"How does TabPFN work?"* — should route to `other` and get a plain answer
- Push the router: find a question that fools it, and look at the probabilities when it does

Or run it from the terminal: `python -m agent "Which customers are most likely to churn?"`

In [ ]:
turn = agent.ask("YOUR QUESTION HERE")
print(turn.answer)

## What we built

| Step | Who decides | Who predicts |
|---|---|---|
| 01 · paste data into an LLM | — | the LLM (guessing) |
| 02 · LLM decides what kind of task it is | the LLM (a label, no confidence) | — |
| 03 · Claude Code with a skill | the LLM, following the skill | TabPFN, via the API |
| 04 · this agent | Jev (a decision model, with probabilities) | TabPFN, via the API |

The pattern generalises past churn. Any time an agent's job touches "what will happen to these rows" — risk, pricing, demand, conversion — the LLM frames and cleans, and a tabular foundation model fits and predicts. The agent's skill is knowing the difference.